In [1]:
pip install Mesa

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install owlready2

     ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
     - -------------------------------------- 0.8/27.4 MB 2.7 MB/s eta 0:00:10
     -- ------------------------------------- 1.6/27.4 MB 3.4 MB/s eta 0:00:08
     --- ------------------------------------ 2.6/27.4 MB 3.8 MB/s eta 0:00:07
     ----- ---------------------------------- 3.7/27.4 MB 4.0 MB/s eta 0:00:06
     ------ --------------------------------- 4.5/27.4 MB 4.0 MB/s eta 0:00:06
     ------- -------------------------------- 5.2/27.4 MB 4.0 MB/s eta 0:00:06
     --------- ------------------------------ 6.3/27.4 MB 4.1 MB/s eta 0:00:06
     ---------- ----------------------------- 7.1/27.4 MB 4.0 MB/s eta 0:00:06
     ----------- ---------------------------- 7.6/27.4 MB 3.8 MB/s eta 0:00:06
     ------------ --------------------------- 8.4/27.4 MB 3.8 MB/s eta 0:00:05
     ------------- -------------------------- 9.4/27.4 MB 3.9 MB/s

In [26]:
!pip install ipywidgets

In [14]:
import random
from typing import List
from mesa import Agent, Model
from owlready2 import (
    World, Thing, ObjectProperty, DataProperty, Not,
    Imp, sync_reasoner_pellet
)

In [15]:
world = World()
onto = world.get_ontology("http://example.org/pizza.owl")

with onto:
    # Domain concepts
    # ... means Ellipsis literal. Still the complete code is not provided, only the definition.
    class Pizza(Thing): ...
    class Topping(Thing): ...
    class MeatTopping(Topping): ...
    class VegTopping(Topping): ...

    class hasTopping(ObjectProperty):
        domain = [Pizza]
        range = [Topping]

    # VegetarianPizza: Pizza but NOT(hasTopping some MeatTopping)
    class VegetarianPizza(Pizza):
        equivalent_to = [Pizza & Not(hasTopping.some(MeatTopping))]
        pass

    # Agent roles
    class AgentThing(Thing): ...
    class Chef(AgentThing): ...
    class Customer(AgentThing): ...
    class Courier(AgentThing): ...
    class Dispatcher(AgentThing): ...

    # Tasks (kept simple; they showcase role semantics)
    class Task(Thing): ...
    class OrderPizzaTask(Task): ...
    class BakePizzaTask(Task): ...
    class DeliverPizzaTask(Task): ...
    class DispatchTask(Task): ...

    class hasTask(ObjectProperty):
        domain = [AgentThing]
        range  = [Task]

    # Chef constraint: vegetarianOnly = True means can bake only vegetarian pizzas
    class vegetarianOnly(DataProperty):
        domain = [Chef]
        range  = [bool]

    # Messages (OWL individuals for semantic communications)
    class Message(Thing): ...
    class OrderRequest(Message): ...
    class BakeOrder(Message): ...
    class DeliveryOrder(Message): ...

    class sender(ObjectProperty):
        domain = [Message]
        range  = [AgentThing]
    class receiver(ObjectProperty):
        domain = [Message]
        range  = [AgentThing]
    class aboutPizza(ObjectProperty):
        domain = [Message]
        range  = [Pizza]
    class quantity(DataProperty):
        domain = [Message]
        range  = [int]
        

In [16]:
onto.save(file="pizza_ontology.owl", format="rdfxml")

In [17]:
#Create toppings and pizzas - Individual creations/mappings
with onto:
    tomato = onto.VegTopping("TomatoSauce")
    mozzarella = onto.VegTopping("Mozzarella")
    basil = onto.VegTopping("Basil")
    pepperoni = onto.MeatTopping("Pepperoni")
    chicken = onto.MeatTopping("Chicken")

    margherita = onto.Pizza("MargheritaPizza")
    # mapping individuals
    margherita.hasTopping = [tomato, mozzarella, basil]

    pepperoni_pizza = onto.Pizza("PepperoniPizza")
    pepperoni_pizza.hasTopping = [tomato, mozzarella, pepperoni]

    bbq_chicken = onto.Pizza("BBQChickenPizza")
    bbq_chicken.hasTopping = [tomato, mozzarella, chicken]

In [18]:
# Create OWL agent individuals
with onto:
    cust1_ind = onto.Customer("cust1")
    cust1_ind.hasTask = [onto.OrderPizzaTask()]

    chefVeg_ind = onto.Chef("chefVeg")
    chefVeg_ind.hasTask = [onto.BakePizzaTask()]
    chefVeg_ind.vegetarianOnly = [True]

    chefAll_ind = onto.Chef("chefAll")
    chefAll_ind.hasTask = [onto.BakePizzaTask()]
    chefAll_ind.vegetarianOnly = [False]

    courier1_ind = onto.Courier("courier1")
    courier1_ind.hasTask = [onto.DeliverPizzaTask()]

    dispatcher_ind = onto.Dispatcher("dispatcher1")
    dispatcher_ind.hasTask = [onto.DispatchTask()]

In [19]:
#SWRL Rules (vegetarian routing)
# If a message is an OrderRequest about a VegetarianPizza, then assign its receiver as chefVeg
with onto:
    rule_route_veg = Imp()

    rule_route_veg.set_as_rule(f"""
        OrderRequest(?m) ^ aboutPizza(?m, ?p) ^ VegetarianPizza(?p) -> receiver(?m, {chefVeg_ind.name})
    """)

In [20]:
#SWRL Rules (non-vegetarian routing)
# If a message is an OrderRequest about a non- VegetarianPizza, then assign its receiver as chefNon-Veg
with onto:
    rule_nonveg = Imp()

rule_nonveg.set_as_rule(f"""
    OrderRequest(?m) ^ aboutPizza(?m, ?p) ^ hasTopping(?p, ?t) ^ MeatTopping(?t)
    -> receiver(?m, {chefAll_ind.name})
""")


# If it’s a pizza that’s **not vegetarian**, send it to the general chef.

OrderRequest(?m), aboutPizza(?m, ?p), hasTopping(?p, ?t), MeatTopping(?t) -> receiver(?m, pizza.chefAll)

In [21]:
#Reasoner helpers
# If needed string param can be provided. (optional)
# load the reasoner.
def try_reason(label:str=""):
    try:
        # infer the propertiese as well, not the classes only
        sync_reasoner_pellet(infer_property_values=True, debug=0)
        if label:
            print(f"[Reasoner] Pellet inference ran ({label}).")
        else:
            print("[Reasoner] Pellet inference ran.")
        return True
    except Exception as e:
        print(f"[Reasoner] Skipped (Pellet/Java not available): {e}")
        return False
# returns true if the pizza_ind param captured is a vegetarian individual
def is_vegetarian(pizza_ind) -> bool:
    try:
        return onto.VegetarianPizza in pizza_ind.is_a
        #in case if reasoner failed, check for pizzas manually which is not with a meat topping
    except Exception:
        return all(not isinstance(t, onto.MeatTopping) for t in pizza_ind.hasTopping)

In [22]:
class BusMessage:
    def __init__(self, owl_ind, mtype: str, sender_id: str, receiver_id: str, pizza_ind, qty: int):
        self.owl_ind = owl_ind
        self.type = mtype          # "order", "bake", "deliver"
        self.sender_id = sender_id
        self.receiver_id = receiver_id
        self.pizza = pizza_ind
        self.qty = qty
        self.processed = False
        self.status = "CREATED"    # will change to ORDERED, ROUTED, BAKING, BAKED, DELIVERED
        self.step_created = 0      # track when created

    def set_status(self, new_status):
        self.status = new_status
        print(f"[STATUS] {self.pizza.name} -> {new_status}")

    def __repr__(self):
        return f"<Msg {self.type} {self.pizza.name} x{self.qty} {self.sender_id}->{self.receiver_id} status={self.status}>"

In [23]:
#MAS model & agents
class PizzaMAS(Model):
    def __init__(self):
        super().__init__()
        self.messages: List[BusMessage] = []
        self.baked = 0
        self.delivered = 0
        self.reasoner_available = False
        # initializing the individuals
        self.cust = cust1_ind
        self.chefVeg = chefVeg_ind
        self.chefAll = chefAll_ind
        self.courier = courier1_ind
        self.dispatcher = dispatcher_ind
        # mapping names for the individuals
        self.agent_customer = CustomerAgent("cust1", self, self.cust)
        self.agent_dispatcher = DispatcherAgent("dispatcher1", self, self.dispatcher)
        self.agent_chefVeg = ChefAgent("chefVeg", self, self.chefVeg)
        self.agent_chefAll = ChefAgent("chefAll", self, self.chefAll)
        self.agent_courier = CourierAgent("courier1", self, self.courier)

        # NEW: create the Manager agent
        self.agent_manager = ManagerAgent("manager1", self, None)   # No OWL individual needed

        self._customer_has_ordered = False
        self.reasoner_available = try_reason("initial")

        print("\\n[Ontology] Vegetarian classification:")
        for p in onto.Pizza.instances():
            print(f" - {p.name:18} vegetarian={is_vegetarian(p)}")

    def send_order(self, sender_name:str, pizza_ind, qty:int=1):
        with onto:
            owl_msg = onto.OrderRequest(f"order_{sender_name}_{random.randrange(1_000_000)}")
            owl_msg.sender = [getattr(onto, sender_name)]
            owl_msg.aboutPizza = [pizza_ind]
            owl_msg.quantity = [qty]
        msg = BusMessage(owl_msg, "order", sender_name, None, pizza_ind, qty)
        msg.status = "ORDERED"                # set status immediately
        msg.step_created = len(self.messages) # track step
        self.messages.append(msg)
        print(f"[MSG] {msg}")

    def send_bake(self, sender_name:str, receiver_name:str, pizza_ind, qty:int):
        with onto:
            owl_msg = onto.BakeOrder(f"bake_{sender_name}_{random.randrange(1_000_000)}")
            owl_msg.sender = [getattr(onto, sender_name)]
            owl_msg.receiver = [getattr(onto, receiver_name)]
            owl_msg.aboutPizza = [pizza_ind]
            owl_msg.quantity = [qty]
        msg = BusMessage(owl_msg, "bake", sender_name, receiver_name, pizza_ind, qty)
        self.messages.append(msg)
        print(f"[MSG] {msg}")

    def send_delivery(self, sender_name:str, receiver_name:str, pizza_ind, qty:int):
        with onto:
            owl_msg = onto.DeliveryOrder(f"deliv_{sender_name}_{random.randrange(1_000_000)}")
            owl_msg.sender = [getattr(onto, sender_name)]
            owl_msg.receiver = [getattr(onto, receiver_name)]
            owl_msg.aboutPizza = [pizza_ind]
            owl_msg.quantity = [qty]
        msg = BusMessage(owl_msg, "deliver", sender_name, receiver_name, pizza_ind, qty)
        self.messages.append(msg)
        print(f"[MSG] {msg}")

    def step(self):
        self.agent_customer.step()
        self.reasoner_available = try_reason("per-step")
        self.agent_dispatcher.step()
        self.agent_chefVeg.step()
        self.agent_chefAll.step()
        self.agent_courier.step()
        self.agent_manager.step()   # Manager report

class OntologyBackedAgent(Agent):
    def __init__(self, unique_id: str, model: PizzaMAS, owl_ind):
        super().__init__(model)
        self.name = unique_id
        self.owl_ind = owl_ind

class CustomerAgent(OntologyBackedAgent):
    def step(self):
        if self.model._customer_has_ordered:
            return
        pizzas = list(onto.Pizza.instances())
        veg = [p for p in pizzas if is_vegetarian(p)]
        pizza = random.choice(veg if veg else pizzas)
        qty = random.choice([1, 2])
        print(f"[Customer] Ordering {pizza.name} x{qty}")
        self.model.send_order(self.name, pizza, qty)
        self.model._customer_has_ordered = True

class DispatcherAgent(OntologyBackedAgent):
    def step(self):
        for msg in self.model.messages:
            if msg.type != "order" or msg.processed:
                continue
            owlr = getattr(msg.owl_ind, "receiver", [])
            if owlr:
                recv_name = owlr[0].name
                print(f"[DISPATCH] SWRL routed {msg.pizza.name} -> {recv_name}")
                self.model.send_bake(self.name, recv_name, msg.pizza, msg.qty)
                msg.set_status("ROUTED")   # NEW
                msg.processed = True
                continue
            recv_name = "chefVeg" if is_vegetarian(msg.pizza) else "chefAll"
            print(f"[DISPATCH] Fallback route {msg.pizza.name} -> {recv_name}")
            self.model.send_bake(self.name, recv_name, msg.pizza, msg.qty)
            msg.set_status("ROUTED")       # NEW
            msg.processed = True

class ChefAgent(OntologyBackedAgent):
    def step(self):
        for msg in self.model.messages:
            if msg.type != "bake" or msg.processed:
                continue
            if msg.receiver_id != self.name:
                continue
            veg_only_vals = getattr(self.owl_ind, "vegetarianOnly", [])
            veg_only = bool(veg_only_vals and veg_only_vals[0])
            if veg_only and not is_vegetarian(msg.pizza):
                print(f"[CHEF {self.name}] Rejects non-veg: {msg.pizza.name}")
                msg.processed = True
                continue
            print(f"[CHEF {self.name}] Baking {msg.pizza.name} x{msg.qty}")
            msg.set_status("BAKING")          # NEW
            self.model.baked += msg.qty
            msg.set_status("BAKED")           # NEW
            self.model.send_delivery(self.name, "courier1", msg.pizza, msg.qty)
            msg.processed = True

class CourierAgent(OntologyBackedAgent):
    def step(self):
        for msg in self.model.messages:
            if msg.type != "deliver" or msg.processed:
                continue
            if msg.receiver_id != self.name:
                continue
            print(f"[COURIER] Delivered {msg.pizza.name} x{msg.qty}")
            self.model.delivered += msg.qty
            msg.set_status("DELIVERED")      # NEW
            msg.processed = True

# --- NEW MANAGER AGENT ---
class ManagerAgent(OntologyBackedAgent):
    def step(self):
        print("\n--- MANAGER REPORT ---")
        for msg in self.model.messages:
            print(f"  {msg}")
        print("Total messages:", len(self.model.messages))
        print("Baked so far:", self.model.baked)
        print("Delivered so far:", self.model.delivered)
        print("----------------------")

In [24]:
#Run
def main():
    model = PizzaMAS()
    print("\\n=== Simulation Start ===")
    for t in range(5):
        print(f"\\n--- Step {t} ---")
        model.step()
    print("\\n=== Summary ===")
    print(f"Baked: {model.baked}, Delivered: {model.delivered}")
    print("\\n[OWL Messages created]")
    for m in onto.Message.instances():
        mtype = m.is_a[0].name if m.is_a else "Message"
        s = m.sender[0].name if getattr(m, "sender", []) else "?"
        r = m.receiver[0].name if getattr(m, "receiver", []) else "?"
        p = m.aboutPizza[0].name if getattr(m, "aboutPizza", []) else "?"
        q = m.quantity[0] if getattr(m, "quantity", []) else "?"
        print(f" - {m.name}: {mtype} {p} x{q} {s}->{r}")

if __name__ == "__main__":
    main()

[Reasoner] Skipped (Pellet/Java not available): Java error message is:
Exception in thread "main" java.lang.UnsupportedClassVersionError: org/apache/jena/riot/lang/LangRDFXML has been compiled by a more recent version of the Java Runtime (class file version 69.0), this version of the Java Runtime only recognizes class file versions up to 66.0
	at java.base/java.lang.ClassLoader.defineClass1(Native Method)
	at java.base/java.lang.ClassLoader.defineClass(ClassLoader.java:1023)
	at java.base/java.security.SecureClassLoader.defineClass(SecureClassLoader.java:150)
	at java.base/jdk.internal.loader.BuiltinClassLoader.defineClass(BuiltinClassLoader.java:862)
	at java.base/jdk.internal.loader.BuiltinClassLoader.findClassOnClassPathOrNull(BuiltinClassLoader.java:760)
	at java.base/jdk.internal.loader.BuiltinClassLoader.loadClassOrNull(BuiltinClassLoader.java:681)
	at java.base/jdk.internal.loader.BuiltinClassLoader.loadClass(BuiltinClassLoader.java:639)
	at java.base/jdk.internal.loader.ClassLo

In [25]:
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: console dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert notebook qtconsole run script server
troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


In [27]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
from collections import Counter
import io, base64

# Create a global model instance and step counter
model = PizzaMAS()
step_count = 0

# Output area for the dashboard
out = widgets.Output()

# Status color mapping
status_colors = {
    "ORDERED": "#FFA500",    # orange
    "ROUTED": "#1E90FF",     # blue
    "BAKING": "#FFD700",     # gold
    "BAKED": "#32CD32",      # lime green
    "DELIVERED": "#00CED1",  # turquoise
    "CREATED": "#D3D3D3"     # light grey
}

def create_status_html(status):
    color = status_colors.get(status, "#FFFFFF")
    return f'<span style="background-color:{color}; padding:2px 8px; border-radius:4px;">{status}</span>'

def update_dashboard():
    """Render the current simulation state as a rich HTML table and charts."""
    with out:
        clear_output(wait=True)
        # Header
        html = f"""
        <h2 style="color:#2c3e50;">🍕 Pizza MAS Dashboard</h2>
        <p><b>Step:</b> {step_count} &nbsp;&nbsp; <b>Baked:</b> {model.baked} &nbsp;&nbsp; <b>Delivered:</b> {model.delivered}</p>
        """
        # Table of messages
        if model.messages:
            html += "<table style='border-collapse: collapse; width:100%;'>"
            html += "<tr style='background-color:#f2f2f2;'><th>Type</th><th>Pizza</th><th>Qty</th><th>Sender</th><th>Receiver</th><th>Status</th></tr>"
            for msg in model.messages:
                status_html = create_status_html(msg.status)
                html += f"""
                <tr style='border-bottom:1px solid #ddd;'>
                    <td>{msg.type}</td>
                    <td>{msg.pizza.name}</td>
                    <td>{msg.qty}</td>
                    <td>{msg.sender_id}</td>
                    <td>{msg.receiver_id or '?'}</td>
                    <td>{status_html}</td>
                </tr>
                """
            html += "</table>"
        else:
            html += "<p>No messages yet.</p>"
        
        # Bar chart of status counts
        status_counts = Counter(msg.status for msg in model.messages)
        if status_counts:
            fig, ax = plt.subplots(figsize=(6, 3))
            colors = [status_colors.get(s, "#999") for s in status_counts.keys()]
            ax.bar(status_counts.keys(), status_counts.values(), color=colors, edgecolor='black')
            ax.set_title("Message Status Counts")
            ax.set_ylabel("Count")
            plt.tight_layout()
            buf = io.BytesIO()
            plt.savefig(buf, format='png')
            buf.seek(0)
            img_data = base64.b64encode(buf.read()).decode('utf-8')
            plt.close()
            html += f'<img src="data:image/png;base64,{img_data}" style="margin-top:10px;"/>'
        
        display(HTML(html))

# Step function
def step_simulation(b):
    global step_count, model
    step_count += 1
    model.step()
    update_dashboard()

# Reset function
def reset_simulation(b):
    global model, step_count
    model = PizzaMAS()
    step_count = 0
    update_dashboard()

# Create widgets
step_button = widgets.Button(description="➡️ Next Step", button_style='primary', icon='step-forward')
reset_button = widgets.Button(description="🔄 Reset", button_style='warning')
auto_slider = widgets.IntSlider(value=1, min=1, max=10, description="Steps to run:")
auto_button = widgets.Button(description="▶️ Run N Steps", button_style='success')

def run_multiple_steps(b):
    global step_count
    n = auto_slider.value
    for _ in range(n):
        step_count += 1
        model.step()
    update_dashboard()

step_button.on_click(step_simulation)
reset_button.on_click(reset_simulation)
auto_button.on_click(run_multiple_steps)

# Arrange widgets
controls = widgets.HBox([step_button, reset_button, auto_slider, auto_button])
dashboard = widgets.VBox([controls, out])

# Initial display
update_dashboard()
display(dashboard)

[Reasoner] Skipped (Pellet/Java not available): Java error message is:
Exception in thread "main" java.lang.UnsupportedClassVersionError: org/apache/jena/riot/lang/LangRDFXML has been compiled by a more recent version of the Java Runtime (class file version 69.0), this version of the Java Runtime only recognizes class file versions up to 66.0
	at java.base/java.lang.ClassLoader.defineClass1(Native Method)
	at java.base/java.lang.ClassLoader.defineClass(ClassLoader.java:1023)
	at java.base/java.security.SecureClassLoader.defineClass(SecureClassLoader.java:150)
	at java.base/jdk.internal.loader.BuiltinClassLoader.defineClass(BuiltinClassLoader.java:862)
	at java.base/jdk.internal.loader.BuiltinClassLoader.findClassOnClassPathOrNull(BuiltinClassLoader.java:760)
	at java.base/jdk.internal.loader.BuiltinClassLoader.loadClassOrNull(BuiltinClassLoader.java:681)
	at java.base/jdk.internal.loader.BuiltinClassLoader.loadClass(BuiltinClassLoader.java:639)
	at java.base/jdk.internal.loader.ClassLo

In [28]:
!pip install streamlit